# RGP-13: 초임계 실유체 모델 — 수정사항 및 사용법

## realFluidFoam-8 → OpenFOAM-13 포팅 결과

| 항목 | 값 |
|------|-----|
| 신규 파일 | 12개 |
| 총 코드 라인 | 3,605줄 |
| 버그 수정 | 3건 (디버거) |
| 성능 최적화 | 12건 (옵티마이저) |
| 기존 OF-13 코드 수정 | 0줄 (완전 플러그인) |

---
## 1. 신규 파일 목록

```
OpenFOAM-13/src/thermophysicalModels/
├── specie/
│   ├── rfSpecie/                               ← 실유체 종 기본 클래스
│   │   ├── rfSpecie.H          (213줄)  Tc,Pc,Vc,omega,miui,kappai,sigmvi 저장
│   │   ├── rfSpecieI.H         (307줄)  인라인 구현, operator 혼합
│   │   └── rfSpecie.C          ( 88줄)  딕셔너리 I/O
│   │
│   ├── equationOfState/SRKGas/                 ← SRK 상태방정식
│   │   ├── SRKGas.H            (278줄)  클래스 선언 (PengRobinsonGas 패턴)
│   │   ├── SRKGasI.H           (579줄)  Z solver, rho, h, Cp, e, Cv 등
│   │   └── SRKGas.C            ( 82줄)  coef1~3 계산, 딕셔너리 I/O
│   │
│   ├── transport/chungTransport/               ← Chung 점성/열전도 + Takahashi 확산
│   │   ├── chungTransport.H    (307줄)  클래스 선언
│   │   ├── chungTransportI.H   (748줄)  mu, kappa, phi, Dimix 구현
│   │   └── chungTransport.C    (222줄)  딕셔너리 I/O
│   │
│   └── include/
│       └── forRealFluidGases.H ( 60줄)  템플릿 인스턴스화 매크로
│
└── multicomponentThermo/mixtures/
    └── SRKchungTakaMixture/                    ← 전용 혼합 규칙
        ├── SRKchungTakaMixture.H  (214줄)  사전계산 행렬 선언
        └── SRKchungTakaMixture.C  (507줄)  혼합규칙 구현
```

---
## 2. OFv8 대비 주요 변경/수정 사항

### 2.1 버그 수정 (디버거)

| # | 파일 | 문제 | 수정 |
|---|------|------|------|
| 1 | `SRKGasI.H` `sp()` | 이상기체 엔트로피 기준항 `-log(p/Pstd)` 누락 | PengRobinsonGas 패턴에 맞게 추가 |
| 2 | `SRKGasI.H` `e()` | OFv8에서 `return 0` → 신규 구현 시 `h-p/rho` 사용했으나 이는 이중 계산 | 출발 함수만 반환하도록 수정: `(T*daAlpha-aAlpha)/b * ln((Z+B)/Z) / W` |
| 3 | `chungTransport.C` `write()` | `this->rfSpecie::name()` 하드코딩 → 템플릿 체인 깨짐 | `this->name()`으로 수정 |

### 2.2 성능 최적화 (옵티마이저)

| # | 파일 | 최적화 | 효과 |
|---|------|--------|------|
| 1 | `SRKGasI.H` | `Xterm()`, `e()`, `Cv()`에서 중복 Z(p,T) 호출 제거 | 셀당 3차 방정식 풀이 3회 절감 |
| 2 | `SRKGasI.H` | `pow(x, 1.0/3.0)` → `cbrt(x)` (4곳) | 하드웨어 명령어 직접 사용 |
| 3 | `SRKGasI.H` | `h()`, `Cp()`에서 `sqrt(T)` 캐싱 | 반복 sqrt 제거 |
| 4 | `chungTransportI.H` | `pow(x, 2.0)` → `sqr(x)`, `pow(x,0.5)` → `sqrt(x)` 등 (~15곳) | pow() 오버헤드 제거 |
| 5 | `chungTransportI.H` `Dimix()` | `T^1.75` 및 `p/101325` 루프 외부 호이스팅 | N-1회 반복 연산 제거 |
| 6 | `SRKchungTakaMixture.C` | 2차 혼합 루프 대칭성 활용 (N² → N(N+1)/2) | N=10일 때 45% 연산량 감소 |
| 7 | `SRKchungTakaMixture.C` | `calculateRealGas(List<scalar>)` → `const List<scalar>&` | 매 셀 리스트 복사 제거 |
| 8 | `SRKchungTakaMixture.C` | `pow(sigma3M, 1/3)` → `cbrt()`, `pow(x,3)` → `x*x*x` 등 | 런타임 pow() 제거 |

### 2.3 OFv8 미구현 → OF-13 신규 구현

| 메서드 | OFv8 | OF-13 (RGP-13) |
|--------|------|----------------|
| `SRKGas::e(p,T)` | `return 0` | 출발 내부에너지 완전 구현 |
| `SRKGas::Cv(p,T)` | `return 0` | `Cp - CpMCv` 기반 구현 |
| `SRKGas::dZdT(p,T,Z)` | 없음 (매번 Z 재계산) | Z 캐시 버전 오버로드 추가 |

### 2.4 OF-13 API 적응

| 항목 | OFv8 | OF-13 |
|------|------|-------|
| 가스상수 | `RR` (전역) | `Foam::constant::thermodynamic::RR` |
| 메서드 대소문자 | `H()`, `E()`, `Sp()` | `h()`, `e()`, `sp()` |
| Specie 기본 클래스 | `rfSpecie` 직접 멤버 접근 | `this->Tc()` 접근자 사용 |
| Mixture 인터페이스 | `cellMixture(celli)` | `thermoMixture(Y)`, `transportMixture(Y)` |
| 혼합물 접근 | `Y_[i][celli]` 직접 | `scalarFieldListSlice` 통해 |

---
## 3. 컴파일 방법

### 3.1 Make/files 수정

**`src/thermophysicalModels/specie/Make/files`** 에 추가:
```make
rfSpecie/rfSpecie.C
equationOfState/SRKGas/SRKGas.C
transport/chungTransport/chungTransport.C
```

**`src/thermophysicalModels/multicomponentThermo/Make/files`** 에 추가:
```make
mixtures/SRKchungTakaMixture/SRKchungTakaMixture.C
```

### 3.2 Make/options 수정

**`src/thermophysicalModels/multicomponentThermo/Make/options`** 의 `EXE_INC`에 추가:
```make
-I$(LIB_SRC)/thermophysicalModels/specie/rfSpecie \
-I$(LIB_SRC)/thermophysicalModels/specie/equationOfState/SRKGas \
-I$(LIB_SRC)/thermophysicalModels/specie/transport/chungTransport
```

### 3.3 빌드

```bash
# Apptainer 컨테이너 내에서 실행
cd $FOAM_SRC/thermophysicalModels/specie && wmake libso
cd $FOAM_SRC/thermophysicalModels/multicomponentThermo && wmake libso
```

---
## 4. 사용법 — 케이스 설정

### 4.1 `constant/physicalProperties`

```cpp
thermoType
{
    type            hePsiThermo;
    mixture         SRKchungTakaMixture;
    transport       chung;
    thermo          janaf;
    equationOfState SRKGas;
    specie          rfSpecie;
    energy          sensibleEnthalpy;
}

species (O2 H2 H2O N2 OH H);

// 각 종별 물성 정의
O2
{
    specie
    {
        molWeight   31.998;
    }
    rfProperties
    {
        Tc          154.58;     // [K]
        Pc          5.043e6;    // [Pa]
        Vc          73.37;      // [cm3/mol]
        omega       0.0222;     // [-]
        miui        0.0;        // [Debye]
        kappai      0.0;        // [-]
        sigmvi      16.6;       // [-] (Fuller diffusion volume)
    }
    thermodynamics
    {
        Tlow        200;
        Thigh       6000;
        Tcommon     1000;
        highCpCoeffs ( ... );   // JANAF 계수
        lowCpCoeffs  ( ... );
    }
    transport
    {
        // chungTransport는 rfProperties에서 자동 계산
        // 추가 입력 불필요
    }
}

H2
{
    specie { molWeight 2.016; }
    rfProperties
    {
        Tc          33.145;
        Pc          1.2964e6;
        Vc          64.2;
        omega       -0.219;
        miui        0.0;
        kappai      0.0;
        sigmvi      7.07;
    }
    thermodynamics { ... }
    transport {}
}

// H2O, N2, OH, H 등 동일 형식으로 정의
```

### 4.2 주요 종별 임계점 물성 참고

| 종 | Tc [K] | Pc [MPa] | Vc [cm³/mol] | omega | sigmvi |
|----|--------|----------|--------------|-------|--------|
| O₂ | 154.58 | 5.043 | 73.37 | 0.0222 | 16.6 |
| H₂ | 33.145 | 1.296 | 64.2 | -0.219 | 7.07 |
| H₂O | 647.14 | 22.064 | 55.95 | 0.3443 | 12.7 |
| N₂ | 126.19 | 3.396 | 89.21 | 0.0372 | 17.9 |
| CH₄ | 190.56 | 4.599 | 98.6 | 0.0115 | 24.42 |
| CO₂ | 304.13 | 7.377 | 94.07 | 0.2236 | 26.9 |
| OH | 400.0 | 8.200 | 50.0 | 0.451 | 10.2 |
| H | 33.0 | 1.280 | 65.0 | -0.22 | 2.31 |

### 4.3 `system/controlDict`

```cpp
application     multicomponentFluid;  // 기존 솔버 그대로 사용

// 초임계 해석 시 권장 설정
deltaT          1e-7;
maxCo           0.3;
```

### 4.4 `system/fvSolution`

```cpp
PIMPLE
{
    nOuterCorrectors    2;      // 초임계 밀도 변화에 대응
    nCorrectors         3;
    nNonOrthogonalCorrectors 1;
    
    // 초임계 물성 안정화
    maxDeltaT       1e-5;
    alphaTemp       0.1;        // 온도 스무딩
    alphaY          0.1;        // 종 스무딩
}
```

---
## 5. 모듈별 수식 요약

### 5.1 SRKGas — 상태방정식

$$P = \frac{RT}{V-b} - \frac{a\alpha(T)}{V(V+b)}$$

$$a\alpha(T) = \mathrm{coef}_1 - \mathrm{coef}_2\sqrt{T} + \mathrm{coef}_3 \cdot T$$

$$Z^3 - Z^2 + (A - B - B^2)Z - AB = 0 \quad (\mathrm{Cardano})$$

### 5.2 chungTransport — 점성/열전도

$$\mu = 0.1\left(\eta_k + \eta_p\right) \quad [\mathrm{kg}/(\mathrm{m}\cdot\mathrm{s})]$$

$$\kappa = 418.68\left(\lambda_k + \lambda_p\right) \quad [\mathrm{W}/(\mathrm{m}\cdot\mathrm{K})]$$

### 5.3 Takahashi — 확산계수

$$D_{i,\mathrm{mix}} = \frac{1 - Y_i}{\sum_{j \neq i} X_j / D_{ij}}, \quad D_{ij} = \phi(P_r, T_r) \cdot \frac{0.001 \cdot T^{1.75} \sqrt{M_{ij}}}{(p/\mathrm{atm}) \cdot \sigma_{ij}^2}$$

### 5.4 혼합규칙 — SRKchungTakaMixture

| 파라미터 | 규칙 | 루프 |
|---------|------|------|
| $b_M$ | 선형: $\sum X_i b_i$ | O(N) |
| coef1~3 | 2차: $\sum\sum X_i X_j \mathrm{COEF}_{ij}$ | O(N²/2) 대칭 |
| $\sigma_M$, $(\epsilon/k)_M$ | 2차 L-J 결합 | O(N²/2) 대칭 |
| $T_{c,ij}$, $P_{c,ij}$ | 산술평균 | O(N²) |

In [4]:
from IPython.display import display, HTML

def render_mermaid(mermaid_code, width="100%", height=None):
    import hashlib
    uid = "m" + hashlib.md5(mermaid_code.encode()).hexdigest()[:8]
    height_style = f"min-height:{height}px;" if height else ""
    html = f"""
    <div id="{uid}" style="width:{width};{height_style}">
        <pre class="mermaid">
{mermaid_code}
        </pre>
    </div>
    <script type="module">
        import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
        mermaid.initialize({{ startOnLoad: true, theme: 'base',
            themeVariables: {{ primaryColor: '#BBDEFB', primaryTextColor: '#1a1a1a',
                primaryBorderColor: '#1565C0', lineColor: '#555', fontSize: '14px' }}
        }});
        mermaid.run({{ nodes: document.querySelectorAll('#{uid} .mermaid') }});
    </script>"""
    display(HTML(html))

render_mermaid("""
flowchart TD
    subgraph BUILD["빌드 순서"]
        direction TB
        B1["<b>1. specie 라이브러리</b>\ncd $FOAM_SRC/.../specie\nwmake libso"]
        B2["<b>2. multicomponentThermo</b>\ncd $FOAM_SRC/.../multicomponentThermo\nwmake libso"]
        B3["<b>3. 솔버 (수정 없음)</b>\nmulticomponentFluid\n기존 바이너리 그대로 사용"]
        B1 --> B2 --> B3
    end

    subgraph TEMPLATE["템플릿 조합 체인"]
        direction LR
        T1["rfSpecie\n(Tc,Pc,Vc,ω,μi,κi,σvi)"]
        T2["SRKGas\n(b, coef1~3, Z solver)"]
        T3["janafThermo\n(JANAF Cp)"]
        T4["chungTransport\n(μ,κ: Chung / D: Takahashi)"]
        T1 --> T2 --> T3 --> T4
    end

    subgraph RUNTIME["런타임 실행 흐름"]
        direction TB
        R1["physicalProperties\nthermoType 지정"]
        R2["SRKchungTakaMixture\n사전계산 행렬 초기화"]
        R3["매 셀마다:\nYi→Xi 변환\ncalculateRealGas()"]
        R4["updateEoS + updateTRANS\n→ rho, mu, kappa, Dimix"]
        R1 --> R2 --> R3 --> R4
    end

    style BUILD fill:#E8F5E9,stroke:#2E7D32
    style TEMPLATE fill:#E3F2FD,stroke:#1565C0
    style RUNTIME fill:#FFF3E0,stroke:#E65100
""")

---
## 6. 다음 단계 (TODO)

### 즉시 필요
- [ ] `Make/files`, `Make/options` 실제 수정 후 `wmake libso` 컴파일 테스트
- [ ] 템플릿 인스턴스화 확인 — `forRealFluidGases.H` 매크로 전개 검증
- [ ] `SRKchungTakaMixture`가 OF-13 `multicomponentMixture` 인터페이스와 정확히 호환되는지 컴파일러 확인
- [ ] 단일종 NIST 검증 (N₂, O₂, CH₄ — ρ, Cp, μ, κ vs 온도/압력)

### 향후
- [ ] 이성분 상호작용 매개변수 $k_{ij}$ 지원 추가
- [ ] 액상 근 선택 로직 (현재는 가스상 최대근만 선택)
- [ ] Oefelein LOx/GH₂ 벤치마크 케이스 구성 및 실행
- [ ] 복사 모델과의 통합 테스트 (P1 + 초임계 물성)